# UTS Deep Learning - Task 1: Multimodal Defect Classification
**Penulis**: ML Engineer Senior  
**Objektif**: Mengklasifikasikan cacat produk manufaktur ke dalam 7 kelas persis: `normal`, `gores`, `noda`, `retak`, `gelembung`, `las_dingin`, `kontaminasi`  
**Metrik Penilaian Utama**: Macro-F1 Score atas 7 kelas  
**Modalitas Input**:  
1. **Kamera Citra**: Permukaan komponen 64x64 piksel monokrom (uint8)  
2. **Sensor Tekan**: Sinyal 3 kanal sepanjang 128 timestep (Kanal 0: Suhu °C, Kanal 1: Getaran relatif, Kanal 2: Arus A)

---
Notebook ini menyajikan pipeline lengkap, hasil eksplorasi data (EDA), perbandingan strategi fusi, analisis nilai modalitas, dan evaluasi empiris siap dipertanggungjawabkan saat ujian lisan.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import Image, display

# Dynamic Root Discovery (Bekerja secara deterministik dari direktori mana pun)
current_dir = os.path.abspath('')
project_root = current_dir
for _ in range(6):
    if os.path.exists(os.path.join(project_root, 'data')) or os.path.exists(os.path.join(project_root, 'artifacts')):
        break
    parent = os.path.dirname(project_root)
    if parent == project_root:
        break
    project_root = parent

cache_dir = os.path.join(project_root, 'data', 'cache')
artifacts_dir = os.path.join(project_root, 'artifacts', 'task1')
src_dir = os.path.join(project_root, 'kode', 'task1_klasifikasi')

if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from src.utils import seed_everything
seed_everything(42)

print("Project Root  :", project_root)
print("PyTorch Versi :", torch.__version__)
print("CUDA Status   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device    :", torch.cuda.get_device_name(0))


## 1. Pemuatan Metadata & Verifikasi Data
Data mentah di-*cache* ke format NumPy dan Parquet di `data/cache/` agar training efisien dan terhindar dari I/O bottleneck.

In [ ]:
with open(os.path.join(cache_dir, 'class_mapping.json')) as f:
    mapping = json.load(f)
class_names = mapping['class_names']
print("7 Kelas Target Persis:", class_names)

train_meta = pd.read_parquet(os.path.join(cache_dir, 'train_meta.parquet'))
test_meta = pd.read_parquet(os.path.join(cache_dir, 'test_meta.parquet'))
print(f"Data Latih: {len(train_meta):,} sampel | Data Uji: {len(test_meta):,} sampel")
train_meta.head()

## 2. Eksplorasi Data (Task a: EDA)
### a. Visualisasi Contoh Tiap Kelas pada Kedua Modalitas (Grid Gambar + Sinyal Sensor)

In [ ]:
fig_a1 = os.path.join(artifacts_dir, 'fig_a1_multimodal_examples.png')
if os.path.exists(fig_a1):
    display(Image(filename=fig_a1))
else:
    print("Figur A1 belum digenerasi.")

### b. Distribusi Kelas & Severe Domain Shift Lini Produksi
Perhatikan bahwa kelas `normal` mendominasi hampir 50% populasi latih. Selain itu, terdapat **domain shift ekstrem**: Lini A mendominasi 75% di data latih, sedangkan pada data uji berbalik menjadi 60% didominasi Lini B.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribusi Kelas
counts = train_meta['kelas'].value_counts()
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, legend=False, ax=axes[0], palette='viridis')
axes[0].set_title("Distribusi 7 Kelas Latih (Ketimpangan Kelas)", fontsize=12, fontweight='bold')
axes[0].set_xticks(range(len(counts)))
axes[0].set_xticklabels(counts.index, rotation=30)
for p in axes[0].patches:
    pct = p.get_height() / len(train_meta) * 100
    axes[0].annotate(f"{int(p.get_height()):,}\n({pct:.1f}%)",
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', va='center', color='white', fontweight='bold')

# Plot 2: Pergeseran Domain Lini Produksi (Train vs Test)
drift_df = pd.DataFrame({
    'Data Latih (Train)': train_meta['lini'].value_counts(normalize=True) * 100,
    'Data Uji (Test)': test_meta['lini'].value_counts(normalize=True) * 100
})
drift_df.plot(kind='bar', ax=axes[1], color=['#2b5c8f', '#e67e22'])
axes[1].set_title("Pembalikan Proporsi Lini Produksi: Train vs Test", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Persentase (%)")
axes[1].set_xticks(range(len(drift_df)))
axes[1].set_xticklabels(drift_df.index, rotation=0)
plt.tight_layout()
plt.show()

### c. Analisis Sensor Hilang (Missing Rate ~5% MCAR)

In [ ]:
fig_a3 = os.path.join(artifacts_dir, 'fig_a3_missing_sensor_rates.png')
if os.path.exists(fig_a3):
    display(Image(filename=fig_a3))


## 3. Sanity Check Optimasi (Task g)
Verifikasi konvergensi satu mini-batch hingga loss mendekati 0 dan akurasi 100% untuk membuktikan stabilitas gradien.

In [ ]:
fig_g1 = os.path.join(artifacts_dir, 'fig_g1_sanity_overfit.png')
if os.path.exists(fig_g1):
    display(Image(filename=fig_g1))


## 4. Perbandingan Strategi Fusi Multimodal (Task b)
Membandingkan 3 strategi integrasi modalitas: Early Concat, Late Decision Fusion, dan Cross-Attention.

In [ ]:
table_b = os.path.join(artifacts_dir, 'table_architecture_comparison.csv')
if os.path.exists(table_b):
    df_b = pd.read_csv(table_b)
    display(df_b)
fig_b = os.path.join(artifacts_dir, 'fig_b1_architecture_comparison.png')
if os.path.exists(fig_b):
    display(Image(filename=fig_b))


## 5. Kontribusi Nilai Tiap Modalitas (Task c: Image-Only vs Sensor-Only vs Multimodal)

In [ ]:
table_c = os.path.join(artifacts_dir, 'table_modality_value.csv')
if os.path.exists(table_c):
    df_c = pd.read_csv(table_c)
    display(df_c)
fig_c1 = os.path.join(artifacts_dir, 'fig_c1_modality_comparison_heatmap.png')
if os.path.exists(fig_c1):
    display(Image(filename=fig_c1))
fig_c2 = os.path.join(artifacts_dir, 'fig_c2_confusion_matrix_multimodal.png')
if os.path.exists(fig_c2):
    display(Image(filename=fig_c2))


## 6. Audit Noisy Labels & Uji Robust Loss (Task d)

In [ ]:
table_d = os.path.join(artifacts_dir, 'table_robust_loss_comparison.csv')
if os.path.exists(table_d):
    display(pd.read_csv(table_d))
fig_d = os.path.join(artifacts_dir, 'fig_d1_loss_distribution_noise.png')
if os.path.exists(fig_d):
    display(Image(filename=fig_d))


## 7. Evaluasi Domain Shift (Task e) & Ketahanan Sensor Hilang (Task f)

In [ ]:
table_e = os.path.join(artifacts_dir, 'table_domain_shift_line_eval.csv')
if os.path.exists(table_e):
    print("--- Performa per Lini Produksi (Domain Shift) ---")
    display(pd.read_csv(table_e))
table_f = os.path.join(artifacts_dir, 'table_missing_modality_eval.csv')
if os.path.exists(table_f):
    print("--- Graceful Degradation Uji Sensor Hilang ---")
    display(pd.read_csv(table_f))
fig_f = os.path.join(artifacts_dir, 'fig_f1_sensor_masking_drop.png')
if os.path.exists(fig_f):
    display(Image(filename=fig_f))


## 8. Model Final 3-Seed Ensemble & Validasi File Submission (Task g)

In [ ]:
table_g = os.path.join(artifacts_dir, 'table_seeds_macro_f1.csv')
if os.path.exists(table_g):
    print("--- Hasil Validasi 3 Seed Berbeda ---")
    display(pd.read_csv(table_g))
fig_g2 = os.path.join(artifacts_dir, 'fig_g2_learning_curves.png')
if os.path.exists(fig_g2):
    display(Image(filename=fig_g2))

sub_path = os.path.join(project_root, 'prediksi_klasifikasi.csv')
if os.path.exists(sub_path):
    sub_df = pd.read_csv(sub_path)
    print(f"\nStatus File Submission: {sub_path}")
    print(f"Jumlah baris: {len(sub_df):,} unit")
    print("Kolom header:", list(sub_df.columns))
    print("Distribusi Prediksi Kelas Data Uji:")
    print(sub_df['kelas'].value_counts())
    display(sub_df.head(10))
else:
    print("File prediksi_klasifikasi.csv sedang digenerasi oleh background training...")
